# PyGeoModel Refactor Test Record

Generated for the `codex/pygeomodel-core-api-refactor` branch.

This notebook records a step-by-step validation of the refactored PyGeoModel package. It separates offline tests, which should always pass, from optional online integration tests that require credentials and OpenGMS network access.

Generated at: 2026-05-10T14:43:35


## 1. Environment and import check

Goal: verify that the refactored package imports from the new `pygeomodel/` package structure and exposes the public API.

In [1]:
from pathlib import Path
import os
import sys
import subprocess

print('Python:', sys.version)
print('CWD:', Path.cwd())
try:
    print('Git branch:', subprocess.check_output(['git', 'branch', '--show-current'], text=True).strip())
    print('Git HEAD:', subprocess.check_output(['git', 'rev-parse', '--short', 'HEAD'], text=True).strip())
except Exception as exc:
    print('Git info unavailable:', exc)

from pygeomodel import (
    GeoModeler,
    ModelService,
    TaskResult,
    RecommendationResult,
    QAResult,
    OpenGMSClient,
)
import pygeomodel

print('pygeomodel module:', pygeomodel.__file__)
print('public imports: OK')


Python: 3.12.6 (main, Jul 12 2025, 14:32:44) [Clang 14.0.3 (clang-1403.0.22.14.1)]
CWD: /Users/mpl/Downloads/coding/project/work/PyGeoModel/tests
Git branch: codex/pygeomodel-core-api-refactor
Git HEAD: 80e1644
pygeomodel module: /Users/mpl/Downloads/coding/project/work/PyGeoModel/pygeomodel/__init__.py
public imports: OK


## 2. Local model catalog loading

Goal: verify that the bundled OpenGMS model catalog can be loaded after moving it into `pygeomodel/data/`.

In [2]:
modeler = GeoModeler()
print('Model count:', len(modeler.model_names))
assert len(modeler.model_names) == 4786, 'Expected 4786 bundled model records'
print('Catalog loading: OK')


Model count: 4786
Catalog loading: OK


## 3. Model search API

Goal: verify `search_models()` returns structured model summaries for a realistic query.

In [3]:
results = modeler.search_models('photovoltaic', limit=5)
for idx, item in enumerate(results, start=1):
    print(f'{idx}. {item.name}')
    print('   description:', (item.description or '')[:180].replace('\n', ' '))

assert results, 'search_models should return at least one photovoltaic-related model'
assert any('Photovoltaic' in item.name for item in results), 'Expected photovoltaic model in results'
print('Model search: OK')


1. Roof Photovoltaic Carbon Emission Reduction Potential Assessment Model
   description: The photovoltaic potential assessment model is a tool for calculating distributed photovoltaic power generation potential at the urban scale. By combining rooftop area data, solar 
Model search: OK


## 4. Model metadata inspection

Goal: verify `get_model()` returns a `ModelService` with parsed inputs, outputs, state names, and data types.

In [4]:
pv_name = 'Roof Photovoltaic Carbon Emission Reduction Potential Assessment Model'
pv_model = modeler.get_model(pv_name)
print('Model:', pv_model.name)
print('Description:', pv_model.description[:300].replace('\n', ' '))
print('Inputs:', len(pv_model.inputs))
print('Outputs:', len(pv_model.outputs))

for item in pv_model.inputs:
    print('INPUT', item.to_dict())
for item in pv_model.outputs:
    print('OUTPUT', item.to_dict())

assert isinstance(pv_model, ModelService)
assert any(item.name == 'system_efficiency' and item.data_type == 'REAL' for item in pv_model.inputs)
assert any(item.name == 'roof_vector_path' and item.is_file for item in pv_model.inputs)
print('Metadata inspection: OK')


Model: Roof Photovoltaic Carbon Emission Reduction Potential Assessment Model
Description: The photovoltaic potential assessment model is a tool for calculating distributed photovoltaic power generation potential at the urban scale. By combining rooftop area data, solar radiation information, and photovoltaic system parameters, it evaluates the theoretical power generation capacity of roo
Inputs: 4
Outputs: 1
INPUT {'state': 'SpatialAnalysis', 'name': 'system_efficiency', 'event_name': 'system_efficiency', 'data_type': 'REAL', 'required': True, 'description': 'System efficiency refers to the overall efficiency of a photovoltaic system, indicating the proportion of energy lost by various components (such as inverters, cables, batteries, etc.) in delivering the electrical energy generated by photovoltaic panels to end users or the grid. It is typically between 0.8 and 0.9.', 'is_file': False, 'children': [{'dataType': 'REAL', 'text': 'system_efficiency', 'desc': 'System efficiency refers

## 5. Parameter normalization without network access

Goal: verify flat Python parameters are converted into the OpenGMS state/event structure, with numeric parameters typed through `dataType` and relative/absolute files treated as files.

In [5]:
from tempfile import TemporaryDirectory
from pathlib import Path

with TemporaryDirectory() as tmpdir:
    roof_file = Path(tmpdir) / 'rooftops.zip'
    roof_file.write_bytes(b'fake rooftop data for parameter-normalization test')
    normalized = pv_model.normalize_params({
        'system_efficiency': '0.8',
        'start_time': '201801',
        'end_time': '201812',
        'roof_vector_path': str(roof_file),
    })

print(normalized)
assert normalized['SpatialAnalysis']['system_efficiency'] == 0.8
assert normalized['SpatialAnalysis']['start_time'] == 201801.0
assert normalized['SpatialAnalysis']['end_time'] == 201812.0
assert normalized['SolarCalculation']['roof_vector_path'].endswith('rooftops.zip')
print('Parameter normalization: OK')


{'SpatialAnalysis': {'system_efficiency': 0.8, 'start_time': 201801.0, 'end_time': 201812.0}, 'SolarCalculation': {'roof_vector_path': '/var/folders/51/pccwx9rs17j2q432g1011yqh0000gn/T/tmp9gdj7fpz/rooftops.zip'}}
Parameter normalization: OK


## 6. Invocation API with a fake OpenGMS client

Goal: verify `invoke()` returns a `TaskResult` and records the normalized state/event input structure without calling OpenGMS.

In [6]:
class FakeClient:
    def __init__(self):
        self.calls = []
        self.manager_url = 'mock://manager'

    def create_task(self, model_name, params, wait=True):
        self.calls.append((model_name, params, wait))
        return {
            'task_id': 'task-123',
            'status': 'completed',
            'outputs': [{
                'statename': 'SolarCalculation',
                'event': 'roofSloar',
                'url': 'http://example.com/result.zip',
                'suffix': 'zip',
                'tag': 'roofSloar',
            }],
            'execution_time': 0.01,
        }

with TemporaryDirectory() as tmpdir:
    roof_file = Path(tmpdir) / 'rooftops.zip'
    roof_file.write_bytes(b'fake rooftop data')
    fake_modeler = GeoModeler(client=FakeClient())
    result = fake_modeler.invoke(
        pv_name,
        params={
            'system_efficiency': 0.8,
            'start_time': 201801,
            'end_time': 201812,
            'roof_vector_path': str(roof_file),
        },
    )

print(result.to_dict())
assert isinstance(result, TaskResult)
assert result.task_id == 'task-123'
assert result.status == 'completed'
assert result.outputs[0]['event'] == 'roofSloar'
assert fake_modeler.last_result is result
print('Invocation with fake client: OK')


{'model_name': 'Roof Photovoltaic Carbon Emission Reduction Potential Assessment Model', 'status': 'completed', 'outputs': [{'statename': 'SolarCalculation', 'event': 'roofSloar', 'url': 'http://example.com/result.zip', 'suffix': 'zip', 'tag': 'roofSloar'}], 'task_id': 'task-123', 'model_id': '7e97e6ce-d968-4fbf-9502-44b5e2c54db8', 'model_md5': '242c696e1162f42d3c239456d354b526', 'params': {'system_efficiency': 0.8, 'start_time': 201801, 'end_time': 201812, 'roof_vector_path': '/var/folders/51/pccwx9rs17j2q432g1011yqh0000gn/T/tmpjnux86nq/rooftops.zip'}, 'uploaded_inputs': {'SpatialAnalysis': {'system_efficiency': 0.8, 'start_time': 201801.0, 'end_time': 201812.0}, 'SolarCalculation': {'roof_vector_path': '/var/folders/51/pccwx9rs17j2q432g1011yqh0000gn/T/tmpjnux86nq/rooftops.zip'}}, 'endpoint': 'mock://manager', 'pygeomodel_version': '1.0.4', 'execution_time': 0.01, 'created_at': 1778395748.416945}
Invocation with fake client: OK


## 7. Structured record serialization

Goal: verify `TaskResult`, `RecommendationResult`, and `QAResult` can be exported as JSON records.

In [7]:
import json
from tempfile import TemporaryDirectory

with TemporaryDirectory() as tmpdir:
    tmpdir = Path(tmpdir)

    task_path = result.to_json(tmpdir / 'execution_record.json')
    rec = RecommendationResult(
        primary_model={'name': pv_name},
        candidates=[{'name': 'candidate'}],
        recommended_data={'local_data': []},
        context={'modeling_history': 'test context'},
    )
    rec_path = rec.to_json(tmpdir / 'recommendation_record.json')

    qa = QAResult(
        question='What input data are required?',
        answer='A rooftop vector dataset and scalar time/efficiency parameters are required.',
        model_name=pv_name,
        sources=[{'type': 'metadata'}],
    )
    qa_path = qa.to_json(tmpdir / 'qa_record.json')

    for path in [task_path, rec_path, qa_path]:
        assert Path(path).exists()
        payload = json.loads(Path(path).read_text(encoding='utf-8'))
        print(path, payload.keys())

print('Record serialization: OK')


/var/folders/51/pccwx9rs17j2q432g1011yqh0000gn/T/tmpj_h_7snt/execution_record.json dict_keys(['model_name', 'status', 'outputs', 'task_id', 'model_id', 'model_md5', 'params', 'uploaded_inputs', 'endpoint', 'pygeomodel_version', 'execution_time', 'created_at'])
/var/folders/51/pccwx9rs17j2q432g1011yqh0000gn/T/tmpj_h_7snt/recommendation_record.json dict_keys(['primary_model', 'candidates', 'recommended_data', 'context', 'trace', 'raw_response'])
/var/folders/51/pccwx9rs17j2q432g1011yqh0000gn/T/tmpj_h_7snt/qa_record.json dict_keys(['question', 'answer', 'model_name', 'rewritten_question', 'sources', 'context', 'raw_response'])
Record serialization: OK


## 8. Recommendation fallback record

Goal: verify `suggest_model()` returns a `RecommendationResult` even without Dify credentials, preserving context and a local metadata-based fallback.

In [8]:
rec = modeler.suggest_model(
    context='Assess rooftop photovoltaic potential in Nanjing using rooftop vector data.',
    data_context='A zipped rooftop polygon dataset is available.',
    include_trace=True,
    return_result=True,
)

print(rec.to_dict())
assert isinstance(rec, RecommendationResult)
assert rec.primary_model
assert 'context' in rec.to_dict()
print('Recommendation fallback: OK')


{'primary_model': {'name': ' Design of Experiment (国产模型培训班专用)', 'description': '实验设计是在参数空间内有效的抽取样本点的过程，目标是使用尽量少的抽样次数尽可能多的充满参数空间。', 'key_strengths': [], 'recommendation_reason': 'Selected from the local OpenGMS model catalog using metadata matching.', 'application_scenario': ''}, 'candidates': [{'name': ' Design of Experiment (国产模型培训班专用)', 'description': '实验设计是在参数空间内有效的抽取样本点的过程，目标是使用尽量少的抽样次数尽可能多的充满参数空间。', 'author': 'qyduan@hhu.edu.cn', 'tags': ['地理信息分析', '地统计分析', '智能计算分析'], 'model_id': '81b1a770-2455-40b3-a992-027ec755055c', 'md5': 'c86fcb1663ae42e8614e27fb482a490a'}, {'name': '1978Texas冬小麦产量区域尺度的估计', 'description': 'YieldEstimation', 'author': 'yue@lreis.ac.cn', 'tags': ['生物过程计算'], 'model_id': '3977b832-ee25-4e11-9cba-184c0c77e136', 'md5': '6d5b39c521ff401cf5be921180e1c151'}, {'name': '24小时中的净长波损失', 'description': 'NetLongwaveLoss', 'author': 'yue@lreis.ac.cn', 'tags': ['地统计分析', '物理过程计算'], 'model_id': '55f1229b-8a10-484e-9dec-ab6ef9472a9c', 'md5': 'cca62581a76cc21b8d82e61f8b8df5b2'}, {

## 9. Q&A metadata fallback

Goal: verify `ask_model()` returns a `QAResult` without requiring OpenAI credentials.

In [9]:
qa = modeler.ask_model(pv_name, 'What input data are required for this model?')
print(qa.to_dict())
assert isinstance(qa, QAResult)
assert qa.answer
assert qa.model_name == pv_name
print('Q&A fallback: OK')


{'question': 'What input data are required for this model?', 'answer': 'Roof Photovoltaic Carbon Emission Reduction Potential Assessment Model is described in the OpenGMS metadata as: The photovoltaic potential assessment model is a tool for calculating distributed photovoltaic power generation potential at the urban scale. By combining rooftop area data, solar radiation information, and photovoltaic system parameters, it evaluates the theoretical power generation capacity of rooftop photovoltaic systems. The model is based on vector areas of rooftops and uses solar radiation data along with meteorological conditions such as cloud cover ratio and atmospheric transmittance to calculate the actual receivable radiation per unit rooftop area over different time periods. Subsequently, the model converts this radiation into usable power generation based on the system efficiency (e.g., 80%). This model is versatile and scalable; it not only supports urban photovoltaic planning and policy-maki

## 10. Notebook widget smoke test

Goal: verify the notebook interface can create widgets. This test does not click Run or call OpenGMS.

In [10]:
widget = modeler.invoke_model(pv_name)
print(type(widget))
print('children:', len(widget.children))
assert hasattr(widget, 'children')
print('Widget smoke test: OK')


<class 'ipywidgets.widgets.widget_box.VBox'>
children: 7
Widget smoke test: OK


## 11. Optional online OpenGMS checks

Goal: check OpenGMS token availability and service status. This section is skipped when `OGMS_TOKEN` is not set.

In [11]:
import os

if not os.environ.get('OGMS_TOKEN'):
    print('SKIPPED: OGMS_TOKEN is not configured. Set OGMS_TOKEN to run online invocation checks.')
else:
    client = OpenGMSClient()
    print('Token valid:', client.validate_token())
    print('PV service ready:', client.check_model_service(pv_name))


SKIPPED: OGMS_TOKEN is not configured. Set OGMS_TOKEN to run online invocation checks.


## 12. Distribution and unit-test commands

Goal: record the command-line checks that should also be run outside the notebook.

In [12]:
import subprocess
from pathlib import Path

cwd = Path.cwd().resolve()
repo_root = cwd if (cwd / 'setup.py').exists() else cwd.parent
assert (repo_root / 'setup.py').exists(), f'Cannot locate repository root from {cwd}'

commands = [
    ['python', '-m', 'unittest', 'discover', '-s', 'tests'],
    [
        'python',
        '-c',
        "from pygeomodel import GeoModeler; m=GeoModeler(); print(len(m.model_names)); print(m.search_models('photovoltaic', limit=1)[0].name)",
    ],
]

for cmd in commands:
    print('$', ' '.join(cmd))
    completed = subprocess.run(
        cmd, cwd=repo_root, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT
    )
    print(completed.stdout)
    assert completed.returncode == 0

print('Command-line checks: OK')


$ python -m unittest discover -s tests


......
----------------------------------------------------------------------
Ran 6 tests in 1.234s

OK

$ python -c from pygeomodel import GeoModeler; m=GeoModeler(); print(len(m.model_names)); print(m.search_models('photovoltaic', limit=1)[0].name)


4786
Roof Photovoltaic Carbon Emission Reduction Potential Assessment Model

Command-line checks: OK


## Summary

If all offline sections pass, the refactored core API is working locally. Any skipped online checks should be revisited after configuring `OGMS_TOKEN` and, if needed, `DIFY_API_KEY` / `OPENAI_API_KEY`.